# RQ1: Query Generation and Retrieval Evaluation

This notebook evaluates the RQ1 runs at three complementary levels: retrieval coverage, ranking after multi-query fusion, and the utility of generated queries. It also separates searches used only for planning from queries whose results were available to the candidate-selection pipeline.

In [ ]:
from pathlib import Path
from collections import defaultdict
from itertools import combinations
import json
import math
import re

import numpy as np
import pandas as pd
import matplotlib
if 'get_ipython' not in globals():
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 100)
plt.rcParams.update({
    'figure.figsize': (9, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / 'gpt_researcher').is_dir() and (path / 'outputs').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the repository.')

OUTPUTS_ROOT = PROJECT_ROOT / 'outputs'
ANALYSIS_DIR = PROJECT_ROOT / 'outputs' / 'evaluation' / 'retrieval'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
K_VALUES = (10, 30, 50, 100, 200, 300)
RRF_K = 60

print(f'Project root: {PROJECT_ROOT}')
print(f'Analysis output: {ANALYSIS_DIR.resolve()}')

## 1. Run configuration and shard merging

Each logical configuration may contain one or more output directories. The B6D2 configuration deliberately combines indices 1-9 from `version2` with indices 10-51 from the resumed run. The earlier incomplete B6D2 directory is not used. Optional wildcard entries allow a future B6D1 resume directory to be incorporated automatically.

In [ ]:
RUN_SPECS = {
    'hybrid_b2_d1': {
        'label': 'Hybrid B2D1', 'breadth': 2, 'depth': 1, 'field_set': 'hybrid',
        'fragments': ['rq1_qwen_hybrid_b2_d1_m2'],
    },
    'hybrid_b4_d1': {
        'label': 'Hybrid B4D1', 'breadth': 4, 'depth': 1, 'field_set': 'hybrid',
        'fragments': ['rq1_qwen_hybrid_b4_d1_m2'],
    },
    'hybrid_b6_d1': {
        'label': 'Hybrid B6D1', 'breadth': 6, 'depth': 1, 'field_set': 'hybrid',
        'fragments': ['rq1_qwen_hybrid_b6_d1_m2', 'rq1_qwen_hybrid_b6_d1_m2_resume*'],
    },
    'hybrid_b4_d2': {
        'label': 'Hybrid B4D2', 'breadth': 4, 'depth': 2, 'field_set': 'hybrid',
        'fragments': ['rq1_qwen_hybrid_b4_d2_m2'],
    },
    'hybrid_b6_d2': {
        'label': 'Hybrid B6D2', 'breadth': 6, 'depth': 2, 'field_set': 'hybrid',
        'fragments': [
            'rq1_qwen_hybrid_b6_d2_m2_version2',
            'rq1_qwen_hybrid_b6_d2_m2_version2_resume_10_51',
        ],
    },
    'retrieval_text_b4_d1': {
        'label': 'Retrieval text B4D1', 'breadth': 4, 'depth': 1, 'field_set': 'retrieval_text',
        'fragments': ['rq1_qwen_retrieval_text_b4_d1_m2'],
    },
    'structured_b4_d1': {
        'label': 'Structured B4D1', 'breadth': 4, 'depth': 1, 'field_set': 'structured',
        'fragments': ['rq1_qwen_structured_b4_d1_m2'],
    },
}

def resolve_fragments(patterns):
    resolved = []
    for pattern in patterns:
        matches = sorted(OUTPUTS_ROOT.glob(pattern))
        if not matches and '*' not in pattern:
            print(f'WARNING: missing run directory: {pattern}')
        for path in matches:
            if path.is_dir() and path not in resolved:
                resolved.append(path)
    return resolved

for key, spec in RUN_SPECS.items():
    spec['paths'] = resolve_fragments(spec['fragments'])
    print(f"{key:24s}: {[p.name for p in spec['paths']]}")

In [ ]:
NCT_RE = re.compile(r'\bNCT\d{8}\b', re.IGNORECASE)

def read_json(path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)

def normalise_nct(value):
    match = NCT_RE.search(str(value or ''))
    return match.group(0).upper() if match else None

def normalise_query(value):
    return ' '.join(str(value or '').split()).strip().casefold()

def artifact_path(run_dir, subdir, pmid, suffix):
    candidates = sorted((run_dir / subdir).glob(f'*_{pmid}_{suffix}'))
    return candidates[-1] if candidates else None

def record_priority(record):
    return (record.get('status') == 'ok', record.get('_fragment_order', -1))

def load_logical_run(run_key, spec):
    merged = {}
    duplicate_rows = []
    for fragment_order, run_dir in enumerate(spec['paths']):
        evaluation_path = run_dir / 'evaluation_run.json'
        if not evaluation_path.exists():
            print(f'WARNING: no evaluation_run.json in {run_dir}')
            continue
        payload = read_json(evaluation_path)
        for row in payload.get('queries', []):
            pmid = str(row.get('pmid'))
            record = dict(row)
            record.update({
                '_run_key': run_key,
                '_run_dir': run_dir,
                '_fragment_order': fragment_order,
                '_query_generation_path': artifact_path(
                    run_dir, 'query_generation', pmid, 'query_generation_trace.json'
                ),
                '_query_map_path': artifact_path(
                    run_dir, 'query_retrieval_maps', pmid, 'query_retrieval_map.json'
                ),
            })
            if pmid in merged:
                duplicate_rows.append((pmid, merged[pmid]['_run_dir'].name, run_dir.name))
                if record_priority(record) <= record_priority(merged[pmid]):
                    continue
            merged[pmid] = record
    return merged, duplicate_rows

LOGICAL_RUNS = {}
DUPLICATES = {}
for run_key, spec in RUN_SPECS.items():
    LOGICAL_RUNS[run_key], DUPLICATES[run_key] = load_logical_run(run_key, spec)

REFERENCE_PMIDS = set(LOGICAL_RUNS['hybrid_b4_d1'])
validation_rows = []
for run_key, records in LOGICAL_RUNS.items():
    ok_records = [r for r in records.values() if r.get('status') == 'ok']
    validation_rows.append({
        'configuration': run_key,
        'fragments': len(RUN_SPECS[run_key]['paths']),
        'reviews_loaded': len(records),
        'reviews_ok': len(ok_records),
        'reviews_failed': len(records) - len(ok_records),
        'missing_pmids': len(REFERENCE_PMIDS - set(records)),
        'query_traces_ok': sum(1 for r in ok_records if r['_query_generation_path']),
        'query_maps_ok': sum(1 for r in ok_records if r['_query_map_path']),
        'duplicates_resolved': len(DUPLICATES[run_key]),
    })

validation_df = pd.DataFrame(validation_rows)
validation_df.to_csv(ANALYSIS_DIR / 'run_validation.csv', index=False)
display(validation_df)

failed_rows = []
for run_key, records in LOGICAL_RUNS.items():
    for pmid, record in records.items():
        if record.get('status') != 'ok':
            failed_rows.append({
                'configuration': run_key, 'pmid': pmid, 'status': record.get('status')
            })
failed_df = pd.DataFrame(failed_rows)
failed_df.to_csv(ANALYSIS_DIR / 'failed_reviews.csv', index=False)
if not failed_df.empty:
    print('Failed reviews that must be rerun or excluded from paired comparisons:')
    display(failed_df)

## 2. Query-role reconstruction and ranking

The original review question is first searched to provide evidence for research-direction planning. It therefore defines the direct-search baseline and is excluded from the agent's retrieval union. Each generated breadth query is subsequently processed by an internal GPT-Researcher instance, which searches both the breadth query and the narrower subqueries produced according to `MAX_ITERATIONS`. All of these non-root query results are available to context construction and are therefore included in the agent evaluation. Follow-up questions and `runs[].input_query` values are planning inputs rather than additional search requests.

The direct-search baseline uses the ranking returned for the original review question. Agent-query rankings are combined using standard Reciprocal Rank Fusion, with one contribution per textually unique query and the best occurrence of an NCT within each query. This prevents an identical breadth query from receiving double weight merely because the internal workflow searches it once while planning subqueries and again while collecting context.

In [ ]:
def deduplicated_ranking(query_map_entry):
    documents = query_map_entry.get('documents_sent_to_gpt_researcher') or []
    if not documents:
        documents = [
            d for d in query_map_entry.get('documents_all', [])
            if d.get('sent_to_gpt_researcher')
        ]
    best_rank = {}
    for position, document in enumerate(documents, start=1):
        nct = normalise_nct(document.get('nct_id'))
        if not nct:
            continue
        rank = document.get('final_rank') or document.get('raw_rank') or position
        try:
            rank = int(rank)
        except (TypeError, ValueError):
            rank = position
        best_rank[nct] = min(best_rank.get(nct, rank), rank)
    return [nct for nct, _ in sorted(best_rank.items(), key=lambda item: (item[1], item[0]))]

def build_map_index(query_map):
    grouped = defaultdict(list)
    for entry in query_map:
        grouped[normalise_query(entry.get('raw_query') or entry.get('search_query'))].append(entry)
    return grouped

def combine_map_entries(entries):
    best_rank = {}
    for entry in entries:
        for rank, nct in enumerate(deduplicated_ranking(entry), start=1):
            best_rank[nct] = min(best_rank.get(nct, rank), rank)
    return [nct for nct, _ in sorted(best_rank.items(), key=lambda item: (item[1], item[0]))]

def recall_at(ranking, ground_truth, k=None):
    gt = set(ground_truth)
    selected = ranking if k is None else ranking[:k]
    return len(gt.intersection(selected)) / len(gt) if gt else np.nan

def r_precision(ranking, ground_truth):
    return recall_at(ranking, ground_truth, len(ground_truth))

def rrf_fuse(rankings, rrf_k=60):
    scores = defaultdict(float)
    best_ranks = {}
    query_hits = defaultdict(int)
    for ranking in rankings:
        seen = set()
        for rank, nct in enumerate(ranking, start=1):
            if nct in seen:
                continue
            seen.add(nct)
            scores[nct] += 1.0 / (rrf_k + rank)
            best_ranks[nct] = min(best_ranks.get(nct, rank), rank)
            query_hits[nct] += 1
    return sorted(
        scores,
        key=lambda nct: (-scores[nct], best_ranks[nct], -query_hits[nct], nct),
    )

def ordered_union(rankings):
    output = []
    seen = set()
    for ranking in rankings:
        for nct in ranking:
            if nct not in seen:
                seen.add(nct)
                output.append(nct)
    return output

def mean_pairwise_jaccard(rankings, k=30):
    sets = [set(r[:k]) for r in rankings if r]
    if len(sets) < 2:
        return np.nan
    values = []
    for left, right in combinations(sets, 2):
        union = left | right
        values.append(len(left & right) / len(union) if union else 0.0)
    return float(np.mean(values))

def token_jaccard(left, right):
    left_tokens = set(re.findall(r'[a-z0-9]+', str(left).casefold()))
    right_tokens = set(re.findall(r'[a-z0-9]+', str(right).casefold()))
    union = left_tokens | right_tokens
    return len(left_tokens & right_tokens) / len(union) if union else np.nan

def reconstruct_review(record, spec):
    if record.get('status') != 'ok':
        return None, []
    if not record.get('_query_generation_path') or not record.get('_query_map_path'):
        return None, []

    generation = read_json(record['_query_generation_path'])
    query_map = read_json(record['_query_map_path'])
    map_index = build_map_index(query_map)
    root_query = generation.get('root_query') or record.get('query')
    root_key = normalise_query(root_query)
    direct_ranking = combine_map_entries(map_index.get(root_key, []))

    descriptors = []
    for run_index, run in enumerate(generation.get('runs', [])):
        depth_remaining = int(run.get('depth') or spec['depth'])
        level = max(1, int(spec['depth']) - depth_remaining + 1)
        for generated_index, generated in enumerate(run.get('generated_queries', [])):
            query_text = generated.get('query')
            query_key = normalise_query(query_text)
            if query_key:
                descriptors.append({
                    'query': query_text,
                    'query_key': query_key,
                    'research_goal': generated.get('researchGoal'),
                    'level': level,
                    'run_index': run_index,
                    'generated_index': generated_index,
                })

    unique_descriptors = []
    seen_queries = set()
    for descriptor in descriptors:
        if descriptor['query_key'] not in seen_queries:
            seen_queries.add(descriptor['query_key'])
            unique_descriptors.append(descriptor)
    descriptor_by_key = {d['query_key']: d for d in unique_descriptors}

    candidate_rankings = []
    level_rankings = defaultdict(list)
    query_rows = []
    gt = {normalise_nct(value) for value in record.get('ground_truth_nct_ids', [])}
    gt.discard(None)
    direct_300 = set(direct_ranking[:300])

    current_descriptor = None
    seen_candidate_keys = set()
    for entry in query_map:
        query_text = entry.get('raw_query') or entry.get('search_query') or ''
        query_key = normalise_query(query_text)
        descriptor = descriptor_by_key.get(query_key)
        if descriptor is not None:
            current_descriptor = descriptor
        if query_key == root_key and descriptor is None:
            continue
        if not query_key or query_key in seen_candidate_keys:
            continue
        seen_candidate_keys.add(query_key)
        entries = map_index.get(query_key, [])
        ranking = combine_map_entries(entries)
        level = (descriptor or current_descriptor or {}).get('level', np.nan)
        query_role = 'breadth_query' if descriptor is not None else 'internal_subquery'
        if ranking:
            candidate_rankings.append(ranking)
            if not pd.isna(level):
                level_rankings[int(level)].append(ranking)
        query_row = {
            'configuration': record['_run_key'],
            'pmid': str(record['pmid']),
            'level': level,
            'query_role': query_role,
            'query': query_text,
            'review_question': root_query,
            'parent_breadth_query': (descriptor or current_descriptor or {}).get('query'),
            'research_goal': (descriptor or current_descriptor or {}).get('research_goal'),
            'query_words': len(query_text.split()),
            'lexical_jaccard_to_review_question': token_jaccard(query_text, root_query),
            'retrieved_count': len(ranking),
            'map_found': bool(entries),
        }
        for k in K_VALUES:
            hits = len(gt.intersection(ranking[:k]))
            query_row[f'gt_hits_at_{k}'] = hits
            query_row[f'hit_at_{k}'] = int(hits > 0)
        query_row['new_gt_vs_direct_at_300'] = len((gt & set(ranking[:300])) - direct_300)
        query_rows.append(query_row)

    union_ranking = ordered_union(candidate_rankings)
    fused_ranking = rrf_fuse(candidate_rankings, RRF_K)

    level_sets = {level: set(ordered_union(ranks)) for level, ranks in level_rankings.items()}
    level_1 = level_sets.get(1, set())
    deeper = set().union(*(values for level, values in level_sets.items() if level > 1)) \
        if any(level > 1 for level in level_sets) else set()

    result = {
        'configuration': record['_run_key'],
        'label': spec['label'],
        'pmid': str(record['pmid']),
        'title': record.get('title') or record.get('review_title') or '',
        'status': record.get('status'),
        'breadth': spec['breadth'],
        'depth': spec['depth'],
        'field_set': spec['field_set'],
        'gt_count': len(gt),
        'direct_available': bool(direct_ranking),
        'generated_query_count': len(descriptors),
        'unique_generated_query_count': len(unique_descriptors),
        'candidate_query_count': len(query_rows),
        'unclassified_candidate_queries': sum(pd.isna(row['level']) for row in query_rows),
        'missing_generated_query_maps': sum(
            descriptor['query_key'] not in map_index for descriptor in unique_descriptors
        ),
        'trace_rows': int(record.get('trace_lines') or record.get('trace_rows_with_nct') or 0),
        'candidate_unique_count': len(union_ranking),
        'candidate_rows_at_300': sum(min(300, len(ranking)) for ranking in candidate_rankings),
        'query_redundancy_jaccard_at_30': mean_pairwise_jaccard(candidate_rankings, 30),
        'union_matched': len(gt & set(union_ranking)),
        'union_recall': recall_at(union_ranking, gt),
        'direct_r_precision': r_precision(direct_ranking, gt),
        'rrf_r_precision': r_precision(fused_ranking, gt),
        'new_gt_vs_direct': len((gt & set(union_ranking)) - direct_300),
        'gt_found_direct': len(gt & direct_300),
        'gt_found_level_1': len(gt & level_1),
        'gt_found_deeper': len(gt & deeper),
        'gt_exclusive_to_deeper': len((gt & deeper) - level_1),
        'baseline_only_gt_not_in_agent': len((gt & direct_300) - set(union_ranking)),
        'reported_actual_recall': record.get('retrieved_actual_recall'),
    }
    if result['reported_actual_recall'] is not None:
        result['reported_actual_recall_difference'] = (
            result['union_recall'] - float(result['reported_actual_recall'])
        )
    else:
        result['reported_actual_recall_difference'] = np.nan
    for k in K_VALUES:
        result[f'direct_recall_at_{k}'] = recall_at(direct_ranking, gt, k)
        result[f'rrf_recall_at_{k}'] = recall_at(fused_ranking, gt, k)
        result[f'query_hit_rate_at_{k}'] = (
            float(np.mean([row[f'hit_at_{k}'] for row in query_rows])) if query_rows else np.nan
        )

    provenance = []
    direct_positions = {nct: rank for rank, nct in enumerate(direct_ranking, 1)}
    fused_positions = {nct: rank for rank, nct in enumerate(fused_ranking, 1)}
    for nct in sorted(gt):
        levels = [level for level, values in level_sets.items() if nct in values]
        provenance.append({
            'configuration': record['_run_key'],
            'pmid': str(record['pmid']),
            'nct_id': nct,
            'direct_rank': direct_positions.get(nct),
            'rrf_rank': fused_positions.get(nct),
            'found_by_candidate_queries': nct in set(union_ranking),
            'first_candidate_level': min(levels) if levels else np.nan,
            'baseline_only_without_agent': (
                nct in direct_300 and nct not in set(union_ranking)
            ),
        })

    return result, query_rows + provenance


## 3. Compute per-review, per-query, and GT-provenance tables

In [ ]:
review_rows = []
query_rows = []
provenance_rows = []

for run_key, records in LOGICAL_RUNS.items():
    spec = RUN_SPECS[run_key]
    for pmid, record in records.items():
        result, mixed_rows = reconstruct_review(record, spec)
        if result is None:
            continue
        review_rows.append(result)
        for row in mixed_rows:
            if 'nct_id' in row:
                provenance_rows.append(row)
            else:
                query_rows.append(row)

reviews_df = pd.DataFrame(review_rows)
queries_df = pd.DataFrame(query_rows)
provenance_df = pd.DataFrame(provenance_rows)

reviews_df.to_csv(ANALYSIS_DIR / 'per_review_metrics.csv', index=False)
queries_df.to_csv(ANALYSIS_DIR / 'per_query_metrics.csv', index=False)
provenance_df.to_csv(ANALYSIS_DIR / 'gt_trial_provenance.csv', index=False)

print(f'Per-review rows: {len(reviews_df)}')
print(f'Per-query rows: {len(queries_df)}')
print(f'GT-provenance rows: {len(provenance_df)}')
display(reviews_df.head())

In [ ]:
def summarise_reviews(frame):
    rows = []
    for configuration, group in frame.groupby('configuration', sort=False):
        spec = RUN_SPECS[configuration]
        gt_total = int(group['gt_count'].sum())
        matched_total = int(group['union_matched'].sum())
        row = {
            'configuration': configuration,
            'label': spec['label'],
            'reviews': len(group),
            'breadth': spec['breadth'],
            'depth': spec['depth'],
            'field_set': spec['field_set'],
            'macro_union_recall': group['union_recall'].mean(),
            'micro_union_recall': matched_total / gt_total if gt_total else np.nan,
            'matched_gt': matched_total,
            'total_gt': gt_total,
            'complete_recall_reviews': int((group['union_recall'] == 1.0).sum()),
            'zero_recall_reviews': int((group['union_recall'] == 0.0).sum()),
            'complete_direct_reviews': int((group['direct_recall_at_300'] == 1.0).sum()),
            'zero_direct_reviews': int((group['direct_recall_at_300'] == 0.0).sum()),
            'macro_direct_recall_at_300': group['direct_recall_at_300'].mean(),
            'mean_breadth_queries_generated': group['unique_generated_query_count'].mean(),
            'mean_unique_agent_queries_searched': group['candidate_query_count'].mean(),
            'mean_trace_rows': group['trace_rows'].mean(),
            'mean_search_requests': group['trace_rows'].mean() / 300.0,
            'mean_unique_candidates': group['candidate_unique_count'].mean(),
            'mean_query_redundancy_at_30': group['query_redundancy_jaccard_at_30'].mean(),
            'new_gt_vs_direct_total': int(group['new_gt_vs_direct'].sum()),
            'baseline_only_gt_total': int(group['baseline_only_gt_not_in_agent'].sum()),
            'max_abs_recall_audit_difference': group['reported_actual_recall_difference'].abs().max(),
        }
        for k in K_VALUES:
            row[f'macro_rrf_recall_at_{k}'] = group[f'rrf_recall_at_{k}'].mean()
        rows.append(row)
    return pd.DataFrame(rows)

summary_available_df = summarise_reviews(reviews_df)
summary_available_df.to_csv(ANALYSIS_DIR / 'run_summary_available_reviews.csv', index=False)
display(summary_available_df.round(4))

pmids_by_config = {
    key: set(reviews_df.loc[reviews_df['configuration'] == key, 'pmid'])
    for key in RUN_SPECS
}
common_pmids = set.intersection(*(values for values in pmids_by_config.values() if values))
paired_reviews_df = reviews_df[reviews_df['pmid'].isin(common_pmids)].copy()
summary_paired_df = summarise_reviews(paired_reviews_df)
summary_paired_df.to_csv(ANALYSIS_DIR / 'run_summary_common_reviews.csv', index=False)
print(f'Common successfully analysed reviews across all configurations: {len(common_pmids)}')
display(summary_paired_df.round(4))

### Complete- and zero-recall reviews

The first table provides compact counts for reporting. The following tables identify the reviews for which the agent retrieved none of the ground-truth trials and show whether the same failures persist across configurations.

In [ ]:
# Use the common-review subset so that every configuration is compared on the same PMIDs.
agent_extremes_df = summary_paired_df[[
    'configuration', 'label', 'reviews',
    'complete_recall_reviews', 'zero_recall_reviews'
]].copy()

reference_configuration = next(iter(RUN_SPECS))
direct_reference = paired_reviews_df[
    paired_reviews_df['configuration'] == reference_configuration
].copy()
direct_extremes_df = pd.DataFrame([{
    'configuration': 'direct_search',
    'label': 'Direct search',
    'reviews': len(direct_reference),
    'complete_recall_reviews': int((direct_reference['direct_recall_at_300'] == 1.0).sum()),
    'zero_recall_reviews': int((direct_reference['direct_recall_at_300'] == 0.0).sum()),
}])
coverage_extremes_df = pd.concat(
    [direct_extremes_df, agent_extremes_df], ignore_index=True
)
coverage_extremes_df['complete_zero'] = (
    coverage_extremes_df['complete_recall_reviews'].astype(str)
    + ' / '
    + coverage_extremes_df['zero_recall_reviews'].astype(str)
)
coverage_extremes_df.to_csv(
    ANALYSIS_DIR / 'coverage_extremes_common_reviews.csv', index=False
)
display(coverage_extremes_df)

zero_recall_reviews_df = (
    paired_reviews_df.loc[
        paired_reviews_df['union_recall'].eq(0),
        [
            'configuration', 'label', 'pmid', 'title', 'gt_count',
            'union_matched', 'candidate_unique_count',
            'gt_found_direct', 'direct_recall_at_300',
            'baseline_only_gt_not_in_agent',
        ],
    ]
    .sort_values(['configuration', 'gt_count', 'pmid'], ascending=[True, False, True])
    .reset_index(drop=True)
)
zero_recall_reviews_df.to_csv(
    ANALYSIS_DIR / 'zero_recall_review_details.csv', index=False
)
display(zero_recall_reviews_df)

zero_recall_overlap_df = (
    zero_recall_reviews_df.groupby(['pmid', 'title', 'gt_count'], as_index=False)
    .agg(
        zero_in_configurations=('configuration', 'nunique'),
        configurations=('label', lambda values: ' | '.join(sorted(set(values)))),
    )
    .sort_values(
        ['zero_in_configurations', 'gt_count', 'pmid'],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
zero_recall_overlap_df.to_csv(
    ANALYSIS_DIR / 'zero_recall_review_overlap.csv', index=False
)
display(zero_recall_overlap_df)

## 4. Paired differences and bootstrap confidence intervals

Differences are calculated per review. The confidence interval is obtained by resampling reviews rather than treating individual NCTs as independent observations.

In [ ]:
PLANNED_COMPARISONS = [
    ('hybrid_b2_d1', 'hybrid_b4_d1', 'Breadth 2 to 4'),
    ('hybrid_b4_d1', 'hybrid_b6_d1', 'Breadth 4 to 6'),
    ('hybrid_b4_d1', 'hybrid_b4_d2', 'Depth 1 to 2 at breadth 4'),
    ('hybrid_b6_d1', 'hybrid_b6_d2', 'Depth 1 to 2 at breadth 6'),
    ('hybrid_b4_d1', 'retrieval_text_b4_d1', 'Hybrid to retrieval text'),
    ('hybrid_b4_d1', 'structured_b4_d1', 'Hybrid to structured'),
    ('structured_b4_d1', 'retrieval_text_b4_d1', 'Structured to retrieval text'),
]

def paired_bootstrap(frame, left, right, metric, n_boot=10000, seed=42):
    pivot = frame.pivot(index='pmid', columns='configuration', values=metric)
    pivot = pivot[[left, right]].dropna()
    differences = (pivot[right] - pivot[left]).to_numpy(dtype=float)
    if not len(differences):
        return None
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(differences), size=(n_boot, len(differences)))
    bootstrap_means = differences[indices].mean(axis=1)
    return {
        'n_reviews': len(differences),
        'left_mean': float(pivot[left].mean()),
        'right_mean': float(pivot[right].mean()),
        'mean_difference': float(differences.mean()),
        'ci_95_low': float(np.quantile(bootstrap_means, 0.025)),
        'ci_95_high': float(np.quantile(bootstrap_means, 0.975)),
        'improved': int((differences > 1e-12).sum()),
        'tied': int((np.abs(differences) <= 1e-12).sum()),
        'worsened': int((differences < -1e-12).sum()),
    }

comparison_rows = []
for left, right, comparison_name in PLANNED_COMPARISONS:
    for metric in ('union_recall', 'rrf_recall_at_100'):
        result = paired_bootstrap(reviews_df, left, right, metric)
        if result:
            result.update({
                'comparison': comparison_name, 'left': left, 'right': right, 'metric': metric
            })
            comparison_rows.append(result)
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(ANALYSIS_DIR / 'paired_comparisons.csv', index=False)
display(comparison_df.round(4))

## 5. Main plots

In [ ]:
def save_figure(name):
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.savefig(ANALYSIS_DIR / f'{name}.pdf', bbox_inches='tight')
    plt.show()
    plt.close()


PLOT_COLOURS = {
    'grey': '#4B5563',
    'light_grey': '#9CA3AF',
    'light_blue': '#56B4E9',
    'blue': '#0072B2',
    'dark_blue': '#004C6D',
    'orange': '#E69F00',
    'green': '#009E73',
}

breadth_keys = ['hybrid_b2_d1', 'hybrid_b4_d1', 'hybrid_b6_d1']
breadth_plot = summary_paired_df.set_index('configuration').loc[breadth_keys].sort_values('breadth')
fig, axis_recall = plt.subplots(figsize=(9, 5))
axis_cost = axis_recall.twinx()
axis_recall.plot(
    breadth_plot['breadth'], breadth_plot['macro_union_recall'],
    marker='o', linewidth=2.5, color=PLOT_COLOURS['blue'], label='Agent union recall'
)
axis_recall.plot(
    breadth_plot['breadth'], breadth_plot['macro_direct_recall_at_300'],
    marker='s', linewidth=2, linestyle='--', color=PLOT_COLOURS['grey'],
    label='Direct search Recall@300'
)
axis_cost.bar(
    breadth_plot['breadth'], breadth_plot['mean_search_requests'], width=0.55,
    color=PLOT_COLOURS['orange'], alpha=0.22, label='Mean index requests'
)
axis_recall.set_xlabel('Research breadth')
axis_recall.set_ylabel('Macro recall')
axis_cost.set_ylabel('Mean index requests (300 results each)')
axis_recall.set_ylim(0, 1.02)
axis_recall.grid(axis='y', alpha=0.25, linestyle='--')
handles_1, labels_1 = axis_recall.get_legend_handles_labels()
handles_2, labels_2 = axis_cost.get_legend_handles_labels()
axis_recall.legend(handles_1 + handles_2, labels_1 + labels_2, loc='lower right')
axis_recall.set_title('Retrieval coverage and query cost by breadth')
save_figure('breadth_recall_and_cost')

In [ ]:
tradeoff_configs = [
    ('hybrid_b2_d1', 'B2D1', PLOT_COLOURS['blue'], 'o', 'Depth 1'),
    ('hybrid_b4_d1', 'B4D1', PLOT_COLOURS['blue'], 'o', None),
    ('hybrid_b6_d1', 'B6D1', PLOT_COLOURS['blue'], 'o', None),
    ('hybrid_b4_d2', 'B4D2', PLOT_COLOURS['green'], 's', 'B4D2 (selected)'),
    ('hybrid_b6_d2', 'B6D2', PLOT_COLOURS['orange'], 's', 'B6D2'),
]
tradeoff_summary = summary_paired_df.set_index('configuration')
direct_coverage = tradeoff_summary.loc['hybrid_b4_d1', 'macro_direct_recall_at_300']

plt.figure(figsize=(9, 5.5))
plt.scatter(
    1.0, direct_coverage, s=105, marker='*', color=PLOT_COLOURS['grey'],
    label='Direct search', zorder=3
)
plt.annotate('Direct', (1.0, direct_coverage), xytext=(7, 7), textcoords='offset points')

for configuration, label, colour, marker, legend_label in tradeoff_configs:
    row = tradeoff_summary.loc[configuration]
    x_value = row['mean_search_requests']
    y_value = row['macro_union_recall']
    plt.scatter(
        x_value, y_value, s=90, marker=marker, color=colour,
        edgecolor='white', linewidth=0.8, label=legend_label, zorder=3
    )
    plt.annotate(label, (x_value, y_value), xytext=(7, 7), textcoords='offset points')

plt.xlabel('Mean index requests per review')
plt.ylabel('Macro coverage')
plt.xlim(left=0)
plt.ylim(0.75, 1.0)
plt.grid(alpha=0.25, linestyle='--')
plt.title('Trade-off between retrieval coverage and search cost')
plt.legend()
save_figure('coverage_cost_tradeoff')

In [ ]:
curve_configs = [
    ('hybrid_b2_d1', 'B2D1', PLOT_COLOURS['light_blue'], 'o', 1.9),
    ('hybrid_b4_d1', 'B4D1', PLOT_COLOURS['blue'], 'o', 2.1),
    ('hybrid_b6_d1', 'B6D1', PLOT_COLOURS['dark_blue'], 'o', 2.1),
    ('hybrid_b4_d2', 'B4D2 (selected)', PLOT_COLOURS['green'], 's', 3.0),
    ('hybrid_b6_d2', 'B6D2', PLOT_COLOURS['orange'], 's', 2.3),
]
plt.figure(figsize=(9, 5.5))
direct_group = paired_reviews_df[paired_reviews_df['configuration'] == 'hybrid_b4_d1']
direct_values = [direct_group[f'direct_recall_at_{k}'].mean() for k in K_VALUES]
plt.plot(
    K_VALUES, direct_values, marker='o', linestyle='--', linewidth=1.9,
    color=PLOT_COLOURS['grey'], label='Direct search'
)
for configuration, label, colour, marker, linewidth in curve_configs:
    group = paired_reviews_df[paired_reviews_df['configuration'] == configuration]
    values = [group[f'rrf_recall_at_{k}'].mean() for k in K_VALUES]
    plt.plot(
        K_VALUES, values, marker=marker, linewidth=linewidth,
        color=colour, label=f'{label} RRF'
    )
plt.xlabel('K unique trials after rank fusion')
plt.ylabel('Macro Recall@K')
plt.ylim(0, 1.02)
plt.grid(alpha=0.25, linestyle='--')
plt.title('Recall at a fixed global candidate budget')
plt.legend(ncol=2)
save_figure('rrf_recall_at_k')

In [ ]:
field_keys = ['hybrid_b4_d1', 'retrieval_text_b4_d1', 'structured_b4_d1']
field_pmids_by_config = {
    key: set(reviews_df.loc[reviews_df['configuration'] == key, 'pmid'])
    for key in field_keys
}
field_common_pmids = set.intersection(*field_pmids_by_config.values())
field_reviews_df = reviews_df[
    reviews_df['configuration'].isin(field_keys)
    & reviews_df['pmid'].isin(field_common_pmids)
].copy()
field_summary_df = summarise_reviews(field_reviews_df)
field_summary_df.to_csv(ANALYSIS_DIR / 'field_set_summary_common_reviews.csv', index=False)
print(f'Common reviews across the three field configurations: {len(field_common_pmids)}')
display(field_summary_df.round(4))

field_plot = field_summary_df.set_index('configuration').loc[field_keys]
x = np.arange(len(field_plot))
width = 0.25
plt.figure(figsize=(9, 5))
plt.bar(
    x - width, field_plot['macro_direct_recall_at_300'], width,
    label='Direct Recall@300', color=PLOT_COLOURS['light_grey']
)
plt.bar(
    x, field_plot['macro_rrf_recall_at_100'], width,
    label='RRF Recall@100', color=PLOT_COLOURS['blue']
)
plt.bar(
    x + width, field_plot['macro_union_recall'], width,
    label='Union recall', color=PLOT_COLOURS['green']
)
plt.xticks(x, ['Hybrid', 'Retrieval text', 'Structured'])
plt.ylabel('Macro recall')
plt.ylim(0, 1.02)
plt.grid(axis='y', alpha=0.25, linestyle='--')
plt.title('End-to-end retrieval by searchable field representation')
plt.legend()
save_figure('field_set_recall_comparison')

field_curve_configs = [
    ('hybrid_b4_d1', 'Hybrid', PLOT_COLOURS['blue'], 'o', 2.2),
    ('retrieval_text_b4_d1', 'Retrieval text (selected)', PLOT_COLOURS['green'], 's', 2.8),
    ('structured_b4_d1', 'Structured fields', PLOT_COLOURS['orange'], '^', 2.2),
]
plt.figure(figsize=(9, 5.5))
for configuration, label, colour, marker, linewidth in field_curve_configs:
    group = field_reviews_df[field_reviews_df['configuration'] == configuration]
    values = [group[f'rrf_recall_at_{k}'].mean() for k in K_VALUES]
    plt.plot(
        K_VALUES, values, marker=marker, linewidth=linewidth,
        color=colour, label=label
    )
plt.xlabel('K unique trials after rank fusion')
plt.ylabel('Macro Recall@K')
plt.ylim(0, 1.02)
plt.grid(alpha=0.25, linestyle='--')
plt.title('Fixed-budget recall by searchable field representation')
plt.legend()
save_figure('field_set_rrf_recall_at_k')

In [ ]:
field_code = {
    'hybrid_b4_d1': 'H',
    'retrieval_text_b4_d1': 'R',
    'structured_b4_d1': 'S',
}
field_provenance = provenance_df[
    provenance_df['configuration'].isin(field_code)
    & provenance_df['pmid'].isin(field_common_pmids)
].copy()
field_provenance['found'] = field_provenance['found_by_candidate_queries'].astype(bool)
field_matrix = field_provenance.pivot_table(
    index=['pmid', 'nct_id'], columns='configuration', values='found', aggfunc='max', fill_value=False
)
for key in field_code:
    if key not in field_matrix:
        field_matrix[key] = False
field_matrix['pattern'] = field_matrix.apply(
    lambda row: ''.join(code for key, code in field_code.items() if bool(row[key])) or 'None', axis=1
)
pattern_labels = {
    'HRS': 'All three configurations',
    'HR': 'Hybrid and retrieval text',
    'HS': 'Hybrid and structured fields',
    'RS': 'Retrieval text and structured fields',
    'H': 'Hybrid only',
    'R': 'Retrieval text only',
    'S': 'Structured fields only',
    'None': 'Not recovered by any configuration',
}
pattern_counts = (
    field_matrix['pattern'].value_counts()
    .rename_axis('field_pattern')
    .reset_index(name='gt_review_trial_pairs')
)
pattern_counts['recovered_by'] = pattern_counts['field_pattern'].map(pattern_labels)
pattern_counts.to_csv(ANALYSIS_DIR / 'field_recovery_patterns.csv', index=False)
display(pattern_counts)

plt.figure(figsize=(8, 4.8))
plot_patterns = pattern_counts.sort_values('gt_review_trial_pairs', ascending=True)
plt.barh(
    plot_patterns['recovered_by'], plot_patterns['gt_review_trial_pairs'],
    color=PLOT_COLOURS['blue']
)
plt.xlabel('Ground-truth review-trial pairs')
plt.ylabel('Recovery pattern')
plt.title('Complementarity of ground-truth recovery across field representations')
save_figure('field_gt_recovery_patterns')

## 6. Query utility and depth contribution

In [ ]:
query_summary = queries_df.groupby(
    ['configuration', 'level', 'query_role'], as_index=False, dropna=False
).agg(
    queries=('query', 'size'),
    mean_query_words=('query_words', 'mean'),
    mean_retrieved_count=('retrieved_count', 'mean'),
    hit_rate_at_30=('hit_at_30', 'mean'),
    hit_rate_at_100=('hit_at_100', 'mean'),
    hit_rate_at_300=('hit_at_300', 'mean'),
    gt_hits_at_300=('gt_hits_at_300', 'sum'),
    new_gt_vs_direct_at_300=('new_gt_vs_direct_at_300', 'sum'),
    missing_maps=('map_found', lambda values: int((~values.astype(bool)).sum())),
)
query_summary.to_csv(ANALYSIS_DIR / 'query_utility_by_level.csv', index=False)
display(query_summary.round(4))

depth_rows = []
for configuration in ('hybrid_b4_d2', 'hybrid_b6_d2'):
    group = paired_reviews_df[paired_reviews_df['configuration'] == configuration]
    depth_rows.append({
        'configuration': configuration,
        'gt_found_direct': group['gt_found_direct'].sum(),
        'gt_found_level_1': group['gt_found_level_1'].sum(),
        'gt_found_deeper': group['gt_found_deeper'].sum(),
        'gt_exclusive_to_deeper': group['gt_exclusive_to_deeper'].sum(),
        'baseline_only_gt_not_in_agent': group['baseline_only_gt_not_in_agent'].sum(),
    })
depth_df = pd.DataFrame(depth_rows)
depth_df.to_csv(ANALYSIS_DIR / 'depth_contribution.csv', index=False)
display(depth_df)


## 7. Qualitative audit of generated breadth queries

Automatic retrieval metrics show whether a query finds known included trials, but they do not establish that its wording is clinically appropriate. The following cell creates a manageable stratified sample of high-utility and zero-hit breadth queries. Review these queries manually for clinical focus, condition or population coverage, intervention coverage, useful terminology expansion, and topic drift. Internal subqueries remain available in `per_query_metrics.csv`, but the primary manual audit targets the breadth queries generated by the deep-research layer.

In [ ]:
breadth_queries = queries_df[queries_df['query_role'] == 'breadth_query'].copy()
manual_samples = []
for configuration, group in breadth_queries.groupby('configuration', sort=False):
    high_value = group.sort_values(
        ['new_gt_vs_direct_at_300', 'gt_hits_at_300', 'hit_at_30'],
        ascending=False,
    ).head(5).copy()
    high_value['sample_reason'] = 'highest retrieval utility'
    zero_hit_pool = group[group['gt_hits_at_300'] == 0]
    if not zero_hit_pool.empty:
        zero_hit = zero_hit_pool.sample(
            n=min(5, len(zero_hit_pool)), random_state=42
        ).copy()
        zero_hit['sample_reason'] = 'zero GT hits at 300'
        manual_samples.append(zero_hit)
    manual_samples.append(high_value)

manual_query_audit_df = pd.concat(manual_samples, ignore_index=True).drop_duplicates(
    ['configuration', 'pmid', 'query']
)
for column in [
    'clinical_focus_ok', 'condition_or_population_present',
    'intervention_present', 'adds_useful_terminology', 'topic_drift', 'notes',
]:
    manual_query_audit_df[column] = ''
manual_query_audit_df.to_csv(
    ANALYSIS_DIR / 'query_examples_for_manual_review.csv', index=False
)
display(manual_query_audit_df[[
    'configuration', 'pmid', 'level', 'sample_reason', 'query',
    'gt_hits_at_30', 'gt_hits_at_300', 'new_gt_vs_direct_at_300',
    'lexical_jaccard_to_review_question',
]].head(20))
